# Phase 3: Fine-tuned RoBERTa

**What we're doing:** Fine-tuning a pre-trained RoBERTa transformer for toxicity classification.

**Why this should beat BiLSTM:**
- **Self-attention**: Every word looks at every other word directly (no sequential bottleneck)
- **Pre-trained knowledge**: RoBERTa was trained on 160GB of text — it already understands English deeply
- **Contextual embeddings**: The same word gets different vectors in different contexts ("bank" of river vs "bank" account)
- **Multi-head attention**: 12 attention heads capture different relationships (negation, syntax, semantics, etc.)

**Goal:** Beat the BiLSTM (Macro F1: 0.5947) by a significant margin.

**Memory strategy for M3 16GB:**
- batch_size = 8 (small to fit in memory)
- gradient_accumulation_steps = 2 (effective batch size = 16)
- max_length = 256 (instead of 512 — saves half the memory)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer

from src.dataset import load_data, get_splits, LABEL_COLS, ToxicDatasetRoBERTa
from src.models import RoBERTaClassifier
from src.training import get_device, FocalLoss, train_roberta, evaluate_roberta
from src.metrics import evaluate_predictions, print_metrics, save_results, load_all_results, print_comparison_table

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

device = get_device()
print(f'PyTorch version: {torch.__version__}')
print('All imports successful!')

---
## Part 1: Load Data

Same splits as before (random_state=42 ensures identical train/val/test sets).

In [ ]:
df = load_data(data_dir='../data')
train_df, val_df, test_df = get_splits(df)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

---
## Part 2: Load RoBERTa Tokenizer

Unlike Phase 2 where we built our own vocabulary, RoBERTa comes with its own pre-trained tokenizer that uses **BPE (Byte-Pair Encoding)**.

BPE splits text into **subwords**. This means:
- No `<UNK>` tokens — any text can be tokenized
- Handles misspellings: `"stoopid"` → `["st", "oop", "id"]` shares pieces with `"stupid"`
- Smaller vocabulary (~50K subwords) but covers more text

In [ ]:
# Load the tokenizer (same one RoBERTa was trained with)
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

print(f'Vocabulary size: {tokenizer.vocab_size:,}')
print(f'Special tokens:')
print(f'  <s>   (start token):   {tokenizer.bos_token_id}')
print(f'  </s>  (end token):     {tokenizer.eos_token_id}')
print(f'  <pad> (padding):       {tokenizer.pad_token_id}')
print(f'  <unk> (unknown):       {tokenizer.unk_token_id}')

In [ ]:
# Demo: see how BPE tokenization works
examples = [
    "You are an idiot",
    "This is a normal sentence about Wikipedia editing",
    "Stoopid misspellings cant fool BPE",
    "f@!#ing creative obfuscation",
]

for text in examples:
    tokens = tokenizer.tokenize(text)
    print(f'Original: "{text}"')
    print(f'Tokens:   {tokens}')
    print(f'Count:    {len(tokens)} tokens\n')

**Notice:**
- `Ġ` prefix means "this token had a space before it" (RoBERTa's space marker)
- `"Stoopid"` gets split into subword pieces — no `<UNK>` needed
- Even creative obfuscation gets tokenized — no information is lost

Compare to Phase 2 where any out-of-vocabulary word became `<UNK>` and lost all meaning.

---
## Part 3: Create PyTorch Datasets and DataLoaders

Same data, but now using `ToxicDatasetRoBERTa` which uses RoBERTa's tokenizer.

Each sample now has THREE tensors:
- `input_ids`: token IDs (256 numbers)
- `attention_mask`: 1 for real tokens, 0 for padding (256 numbers)
- `labels`: 6 toxicity labels

In [ ]:
MAX_LENGTH = 256       # tokens per comment (covers ~90% of comments)
BATCH_SIZE = 8         # small batches to fit in M3 memory
ACCUMULATION_STEPS = 2 # effective batch size = 8 × 2 = 16

train_dataset = ToxicDatasetRoBERTa(train_df, tokenizer, max_length=MAX_LENGTH)
val_dataset = ToxicDatasetRoBERTa(val_df, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training: {len(train_dataset):,} samples → {len(train_loader):,} batches of {BATCH_SIZE}')
print(f'Validation: {len(val_dataset):,} samples → {len(val_loader):,} batches of {BATCH_SIZE}')
print(f'Effective batch size (with accumulation): {BATCH_SIZE * ACCUMULATION_STEPS}')

# Peek at a batch
batch = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  input_ids:      {batch["input_ids"].shape}')
print(f'  attention_mask: {batch["attention_mask"].shape}')
print(f'  labels:         {batch["labels"].shape}')

---
## Part 4: Create the RoBERTa Model

We load pre-trained `roberta-base` and add a classification head on top.

**Architecture:**
```
tokenized text → RoBERTa (12 layers) → [CLS] vector (768 dims) → dropout → linear → 6 logits
```

All ~125M parameters are trainable — we update everything during fine-tuning.

In [ ]:
model = RoBERTaClassifier(num_labels=6, model_name='roberta-base', dropout=0.1)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model loaded on {device}')
print(f'  Total parameters:     {total:,}')
print(f'  Trainable parameters: {trainable:,}  (full fine-tuning)')
print(f'\nFor comparison:')
print(f'  TF-IDF + LogReg:    300,006')
print(f'  BiLSTM + GloVe:     632,326 trainable')
print(f'  RoBERTa-base:    {trainable:>11,}  ← {trainable // 632326}× more!')

---
## Part 5: Setup Training

Key settings:
- **Loss**: Focal Loss (γ=2) — same as Phase 2, handles class imbalance
- **Optimizer**: **AdamW** with lr=2e-5 — much lower than Phase 2 (0.0005) because we're fine-tuning, not training from scratch. Big learning rate would destroy pre-trained knowledge.
- **Weight decay**: 0.01 — standard for transformer fine-tuning
- **Epochs**: only 3 — transformers learn fast and overfit fast

In [ ]:
loss_fn = FocalLoss(alpha=1.0, gamma=2.0)

# AdamW = Adam + proper weight decay (better for transformers)
# lr=2e-5 = 0.00002 — the standard fine-tuning learning rate for BERT/RoBERTa
# Why so small? RoBERTa weights are already good; we just nudge them.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01,
)

# Linear warmup + linear decay scheduler
# Standard for transformer fine-tuning
from transformers import get_linear_schedule_with_warmup

n_epochs = 3
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * n_epochs
warmup_steps = int(0.1 * total_steps)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Loss: Focal Loss (gamma=2.0)')
print(f'Optimizer: AdamW (lr=2e-5, weight_decay=0.01)')
print(f'Scheduler: Linear warmup ({warmup_steps} steps) + linear decay')
print(f'Epochs: {n_epochs}')

---
## Part 6: Train!

**This will take a while** — ~30-60 minutes on M3 for 3 epochs.

Watch the train and val loss decrease. With pre-trained models, the first epoch usually gives most of the improvement.

In [ ]:
history = train_roberta(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    n_epochs=n_epochs,
    patience=2,
    accumulation_steps=ACCUMULATION_STEPS,
    scheduler=scheduler,
    save_path='../models/roberta.pt',
)

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))
epochs = range(1, len(history['train_loss']) + 1)
ax.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax.plot(epochs, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('RoBERTa Fine-tuning Curves')
ax.legend()
plt.tight_layout()
plt.savefig('../results/roberta_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 7: Evaluate with Threshold Tuning

Same approach as Phase 2 — try several thresholds and pick the best.

In [ ]:
val_results = evaluate_roberta(model, val_loader, loss_fn, device)

print('=== THRESHOLD TUNING ===')
print(f'{"Threshold":>10s} | {"Macro F1":>8s} | {"Macro AUC":>9s}')
print('-' * 35)

best_thresh = 0.5
best_f1 = 0
for thresh in [0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (val_results['probabilities'] >= thresh).astype(int)
    results = evaluate_predictions(val_results['labels'], preds, val_results['probabilities'])
    f1 = results['macro']['f1']
    marker = ' ← best' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
    print(f'{thresh:>10.2f} | {f1:>8.4f} | {results["macro"]["roc_auc"]:>9.4f}{marker}')

print(f'\nBest threshold: {best_thresh}')

# Final evaluation with best threshold
final_preds = (val_results['probabilities'] >= best_thresh).astype(int)
roberta_results = evaluate_predictions(
    y_true=val_results['labels'],
    y_pred=final_preds,
    y_proba=val_results['probabilities'],
)

print_metrics(roberta_results, f'RoBERTa fine-tuned (threshold={best_thresh})')

In [ ]:
# Save results
save_results(roberta_results, 'roberta', results_dir='../results')

---
## Part 8: Three-Model Comparison (the resume moment!)

All three models, side by side.

In [ ]:
all_results = load_all_results(results_dir='../results')
print_comparison_table(all_results)

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Order: baseline → BiLSTM → RoBERTa
model_order = ['tfidf_logreg', 'bilstm_glove', 'roberta']
results_dict = {r['model']: r for r in all_results}
ordered_results = [results_dict[m] for m in model_order if m in results_dict]

x = np.arange(len(LABEL_COLS))
width = 0.25
colors = ['steelblue', 'coral', 'forestgreen']
model_names = ['TF-IDF + LogReg', 'BiLSTM + GloVe', 'RoBERTa']

for i, (result, name, color) in enumerate(zip(ordered_results, model_names, colors)):
    f1s = [result['per_label'][l]['f1'] for l in LABEL_COLS]
    ax.bar(x + (i - 1) * width, f1s, width, label=name, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(LABEL_COLS, rotation=45, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score per Label: All 3 Models')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/three_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 9: Test the Model on Examples

See how RoBERTa handles the same examples we tested before.

In [ ]:
examples = [
    "You are a complete idiot and should be banned",
    "Thank you for your contributions to this article",
    "This is not a bad edit at all, well done",
    "I will find you and destroy everything you care about",
    "Please stop removing content from pages",
]

model.eval()
for text in examples:
    encoding = tokenizer(text, max_length=MAX_LENGTH, padding='max_length',
                          truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    print(f'\n"{text}"')
    for label, prob in zip(LABEL_COLS, probs):
        bar = '█' * int(prob * 30)
        flag = ' ← FLAGGED' if prob > best_thresh else ''
        print(f'  {label:>15s}: {prob:.3f} {bar}{flag}')

---
## Summary

Phase 3 results will appear above after training completes.

In [ ]:
print('=' * 50)
print('  PHASE 3 SUMMARY: Fine-tuned RoBERTa')
print('=' * 50)
print(f'  Macro F1:  {roberta_results["macro"]["f1"]:.4f}')
print(f'  Macro AUC: {roberta_results["macro"]["roc_auc"]:.4f}')
print(f'  Threshold: {best_thresh}')
print(f'  Epochs: {len(history["train_loss"])}')
print('=' * 50)

print(f'\n--- Improvement story ---')
print(f'  TF-IDF baseline: 0.5633')
print(f'  BiLSTM + GloVe:  0.5947  (+5.6%)')
print(f'  RoBERTa:        {roberta_results["macro"]["f1"]:.4f}  (+{(roberta_results["macro"]["f1"]-0.5633)/0.5633*100:.1f}% vs baseline)')

---
## Learning Journal

**What I learned in Phase 3:**

- 
- 
- 
- 
- 

---
## Interview Prep Questions

1. **"Explain self-attention in your own words."**
   - *Hint: every word looks at every other word and decides what's important via Q/K/V*

2. **"Why RoBERTa over BERT?"**
   - *Hint: same architecture, better training (more data, no NSP, dynamic masking)*

3. **"How does fine-tuning differ from training from scratch?"**
   - *Hint: starting weights, smaller learning rate, fewer epochs*

4. **"How did you handle memory constraints on 16GB?"**
   - *Hint: smaller batch_size + gradient accumulation, max_length=256*

5. **"What's the difference between BPE tokenization and your TF-IDF vocabulary?"**
   - *Hint: subwords vs whole words, no UNK, handles misspellings*